# Week (2) Module D: Feature Selection, Regression & Model Evaluation
**Course:** Statistics and Machine Learning

---

### Module Contributor(s)
| Full Name | Role |
|---|---|
| `_______________` | `_______________` |
| `_______________` | `_______________` |

---

### About this Module
This module covers **Tasks 10, 11, and 12** of the Week (2) practical.
Your outputs will be merged with **Modules A, B, and C into the group's final notebook**.

**Dataset:** Pima Indians Diabetes → DOI: https://www.openml.org/d/37

| Feature | Description |
|---|---|
| `Pregnancies` | Number of pregnancies |
| `Glucose` | Plasma glucose concentration |
| `BloodPressure` | Diastolic blood pressure (mm Hg) |
| `SkinThickness` | Triceps skin fold thickness (mm) |
| `Insulin` | 2-hour serum insulin (mu U/ml) |
| `BMI` | Body mass index |
| `DiabetesPedigreeFunction` | Genetic risk score |
| `Age` | Age in years |
| `Outcome` | 0 = Non-diabetic, 1 = Diabetic |

---

**Background: Week 2 Day 2 lecture recap:**

Day 2 bridges the gap between data preparation (Day 1) and model training. Three core ideas:

> - **Feature selection:** choosing the most informative subset of features reduces overfitting and improves generalisation. Three families: *Filter* (statistical tests, e.g. correlation, ANOVA F-test), *Wrapper* (recursive elimination, forward selection), *Embedded* (Lasso, Ridge, tree importance).
> - **Train/Test split:** the model is always evaluated on data it has NEVER seen during training. Fitting and evaluating on the same data gives optimistically biased metrics. A held-out test set gives an honest estimate of generalisation performance.
> - **Regression:** supervised learning approach for predicting a *continuous* target. Simple Linear Regression (one feature), Multiple Linear Regression (several features), Polynomial Regression (curved relationships). Evaluation uses MAE, MSE, RMSE (error in target units), and R² (fraction of variance explained).

---
> Cells marked `# TODO` require code. Cells marked `✍️` require a written response.

---
# Task 10: Feature Selection, Train / Test Split & K-Fold Cross-Validation
Not all features are equally useful. Selecting only the most informative ones reduces noise, speeds up training, and improves generalisation.

> **Goal:** Apply filter-based feature selection (Pearson correlation + ANOVA F-test) and a wrapper method (RFE) to the Diabetes dataset, create a clean train/test split, and compare it against 5-Fold Cross-Validation.

In [ ]:
# ── Task 10 Setup (do not modify) ─────────────────────────────────────────────

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# sklearn: the standard Python machine learning library
from sklearn.linear_model import LinearRegression  # OLS linear regression
from sklearn.feature_selection import RFE          # Recursive Feature Elimination (wrapper method)
from sklearn.model_selection import train_test_split  # splits data into train and test sets
from sklearn.preprocessing import StandardScaler      # z-score standardisation

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 4)

URL = 'https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv'
df_raw = pd.read_csv(URL)

# Apply the same cleaning pipeline established in Module C:
# Replace impossible zeros → NaN → median imputation.
SUSPECT_COLS = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df = df_raw.copy()
df[SUSPECT_COLS] = df[SUSPECT_COLS].replace(0, np.nan)
df[SUSPECT_COLS] = df[SUSPECT_COLS].fillna(df[SUSPECT_COLS].median())

FEATURE_COLS = ['Pregnancies','Glucose','BloodPressure','SkinThickness',
                'Insulin','BMI','DiabetesPedigreeFunction','Age']

X = df[FEATURE_COLS]   # all 8 features, shape (768, 8)
y = df['Outcome']      # binary target, shape (768,)

print('Ready')


In [ ]:
# Provided: Filter method: rank features by Pearson |r| with the target

# --- Filter Method: Pearson Correlation ---
# Filter methods select features using statistical relationships computed BEFORE
# model training: they are fast and model-agnostic.
# Here we rank every feature by its absolute Pearson correlation with Outcome.
# Features with |r| closer to 1 have a stronger linear relationship with the target.

corr_scores = {}
for col in FEATURE_COLS:
    # stats.pearsonr returns (r, p_value); we only need |r| for ranking here.
    r, p = stats.pearsonr(X[col], y)
    corr_scores[col] = {'|r| with Outcome': round(abs(r), 4), 'p-value': round(p, 6)}

corr_df = pd.DataFrame(corr_scores).T.sort_values('|r| with Outcome', ascending=False)

print('--- Filter Method: Pearson |r| with Outcome ---')
print(corr_df.to_string())

# Select the top 4 features by |r|: a common heuristic for this dataset size.
TOP_4 = corr_df.head(4).index.tolist()
print(f'\nTop 4 features by Pearson |r|: {TOP_4}')

# Visualise as a bar chart so the ranking is immediately clear.
fig, ax = plt.subplots(figsize=(8, 4))
corr_df['|r| with Outcome'].plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Feature Ranking by Pearson |r| with Outcome (Filter Method)')
ax.set_ylabel('|Pearson r|')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()


In [ ]:
# Provided: Wrapper method: Recursive Feature Elimination (RFE)
#
# RFE is called a 'wrapper' method because it WRAPS a machine learning model
# inside the feature selection loop. It uses the model's own sense of feature
# importance, not a generic statistic like Pearson r.
#
# How RFE works (from the lecture, slide on Wrapper Methods):
#   Step 1  Start with all features.
#   Step 2  Train the model on the full feature set.
#   Step 3  Rank features by importance (for LinearRegression: by |coefficient|).
#   Step 4  Remove the LEAST important feature.
#   Step 5  Retrain the model and repeat until n_features_to_select remain.

from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

# Step 1: Scale features BEFORE RFE.
# RFE ranks by |coefficient|. If Insulin (range 0–800) is not scaled,
# its coefficient will be tiny just because of its large scale, not
# because it is unimportant. Scaling levels the playing field.
scaler_rfe = StandardScaler()
X_scaled   = scaler_rfe.fit_transform(X)   # fit on ALL data for feature selection only
# Note: we fit on all data here because RFE is for selection, not evaluation.
# The train/test split (for honest evaluation) comes in the next cell.

# Step 2: Create the RFE object.
# estimator = the model used to judge feature importance.
# n_features_to_select = how many features to keep.
# step = how many features to remove per iteration (1 = slowest but most precise).
estimator = LinearRegression()
rfe       = RFE(estimator=estimator, n_features_to_select=4, step=1)

# Step 3: Fit RFE, this runs the full elimination loop internally.
rfe.fit(X_scaled, y)

# Step 4: Inspect results.
# rfe.ranking_ gives each feature a rank: rank=1 means SELECTED; higher = eliminated earlier.
# rfe.support_ gives True for selected features, False for eliminated ones.
rfe_results = pd.DataFrame({
    'Feature':   FEATURE_COLS,
    'RFE Rank':  rfe.ranking_,
    'Selected?': rfe.support_
}).sort_values('RFE Rank')

print('--- Wrapper Method: Recursive Feature Elimination (RFE) ---')
print(f'Target: keep the top {rfe.n_features_to_select_} features out of {len(FEATURE_COLS)}')
print()
print(rfe_results.to_string(index=False))

RFE_TOP4 = rfe_results[rfe_results['Selected?']]['Feature'].tolist()
print(f'\nRFE selected features: {RFE_TOP4}')

# Step 5: Compare Pearson filter vs RFE wrapper.
print(f'\nPearson top-4 : {TOP_4}')
print(f'RFE top-4     : {RFE_TOP4}')
agreed = set(TOP_4) == set(RFE_TOP4)
print(f'Agreement     : {"Both methods agree" if agreed else " Methods disagree: see Interpretation cell"}')


In [ ]:
# Provided: Wrapper method: Forward Feature Selection
#
# Forward selection is the OPPOSITE of RFE:
# it starts with NO features and adds the most useful one at a time.
# From the lecture (Week 2 Day 2 slides):
#   Step 1 Start with an empty feature set.
#   Step 2 Test each remaining feature individually.
#   Step 3 Add the feature that improves model score the most.
#   Step 4 Repeat until n_features_to_select features have been added.
#
# sklearn provides SequentialFeatureSelector for this purpose.

from sklearn.feature_selection import SequentialFeatureSelector

# SequentialFeatureSelector with direction='forward' = forward selection.
# cv=5: uses 5-fold cross-validation to score each candidate feature set.
# scoring='r2': the feature that maximises R² is added at each step.
sfs = SequentialFeatureSelector(
    estimator=LinearRegression(),
    n_features_to_select=4,
    direction='forward',    # 'forward' adds features; 'backward' removes (= greedy RFE)
    scoring='r2',
    cv=5
)

# We pass X (unscaled), SequentialFeatureSelector uses CV internally,
# so scaling would need to be inside a Pipeline for strict correctness.
# For feature selection comparison purposes, this is sufficient.
sfs.fit(X, y)

# .get_support() returns a boolean mask aligned with FEATURE_COLS.
sfs_mask = sfs.get_support()
SFS_TOP4 = [feat for feat, sel in zip(FEATURE_COLS, sfs_mask) if sel]

print('--- Wrapper Method: Forward Feature Selection ---')
sfs_results = pd.DataFrame({'Feature': FEATURE_COLS, 'Selected?': sfs_mask})
print(sfs_results.to_string(index=False))
print(f'\nForward selection chose: {SFS_TOP4}')

# Summary across all three filter/wrapper approaches so far.
print('\n=== Feature Selection Summary ===')
print(f'Pearson |r|       : {TOP_4}')
print(f'RFE (backward)    : {RFE_TOP4}')
print(f'Forward selection : {SFS_TOP4}')


In [ ]:
# Provided: Embedded method Lasso Regression (L1 regularisation)
#
# Embedded methods perform feature selection DURING model training.
# From the lecture (Week 2 Day 2):
#   - Lasso (L1): adds a penalty = α × Σ|βⱼ| to the loss.
#     Features with low predictive value get their coefficient shrunk to EXACTLY zero.
#     Coefficient = 0 means the feature is effectively removed from the model.
#   - Ridge (L2): adds a penalty = α × Σβⱼ². Shrinks coefficients toward zero
#     but never to exactly zero, Ridge does NOT perform feature selection.
#
# This is why Lasso is the standard embedded feature selection tool:
# it selects features and fits the model in one step.

from sklearn.linear_model import Lasso
from sklearn.pipeline import Pipeline

# We use a Pipeline to chain scaling and Lasso prevents data leakage.
# alpha (λ in the lecture notation) controls regularisation strength:
#   α = 0   → plain OLS (no penalty, no selection)
#   α large → all coefficients driven to zero (too aggressive)
# We try three values to show the effect.

print('--- Embedded Method: Lasso (L1) Feature Selection ---')
print(f'{"Alpha (λ)":<12}', end='')
for feat in FEATURE_COLS:
    print(f'{feat[:10]:<12}', end='')
print('Features kept')
print('-' * (12 + 12*len(FEATURE_COLS) + 14))

for alpha in [0.01, 0.1, 1.0]:
    lasso_pipe = Pipeline([
        ('scaler', StandardScaler()),    # fit scaler on training data (X here for selection)
        ('lasso',  Lasso(alpha=alpha, max_iter=5000))
    ])
    lasso_pipe.fit(X, y)

    coefs  = lasso_pipe.named_steps['lasso'].coef_
    # A coefficient is 'zeroed out' if its absolute value is below a tiny threshold.
    # Floating point arithmetic rarely gives exactly 0.0, so we use 1e-6 as a threshold.
    kept   = (np.abs(coefs) > 1e-6).sum()
    print(f'{alpha:<12}', end='')
    for c in coefs:
        marker = f'{c:.3f}' if abs(c) > 1e-6 else '  ZERO  '
        print(f'{marker:<12}', end='')
    print(f'{kept}/{len(FEATURE_COLS)}')

print()
print('Observation: as alpha increases, Lasso zeros out more coefficients.')
print('Features zeroed at alpha=0.1 are the ones Lasso considers least useful.')
print('Compare these eliminated features with the Pearson and RFE rankings above.')


In [ ]:
# TODO: Build a feature selection comparison table across all three method families.
#
# You have now seen:
#   - Filter  : Pearson |r| → ranked by TOP_4
#   - Wrapper : RFE → ranked by RFE_TOP4, Forward selection → SFS_TOP4
#   - Embedded: Lasso → features zeroed at alpha=0.1
#
# Steps:
#   1. For each of the 8 FEATURE_COLS, create a row with:
#        'Feature' | 'Pearson |r|' | 'Pearson selected?' | 'RFE rank' |
#        'RFE selected?' | 'SFS selected?' | 'Lasso coef (α=0.1)' | 'Lasso selected?'
#
#   2. To get the Lasso coefficients at alpha=0.1, re-fit once:
#        lasso_01 = Pipeline([('sc', StandardScaler()), ('m', Lasso(alpha=0.1, max_iter=5000))])
#        lasso_01.fit(X, y)
#        lasso_coefs = lasso_01.named_steps['m'].coef_
#
#   3. Sort the table by Pearson |r| descending.
#   4. Print the full table.
#   5. Count how many features ALL three method families agree on as 'selected'.
#      Print: 'All three methods agree on N features: [...]'

# YOUR CODE HERE


In [ ]:
# TODO: Create the official train/test split for use in Tasks 11 and 12.
#
# Why split BEFORE any preprocessing or model fitting?
# If we compute scaling parameters (mean, std) on the full dataset and THEN split,
# test information 'leaks' into the training process, the model effectively 'saw'
# the test data. This is called DATA LEAKAGE and leads to overoptimistic metrics.
# Correct order: SPLIT FIRST → fit scaler on train only → transform both sets.
#
# Steps:
#   1. Use train_test_split(X, y, test_size=0.20, random_state=42, stratify=y).
#      stratify=y preserves the class ratio in both sets (important for imbalanced data).
#   2. Store as X_train, X_test, y_train, y_test.
#   3. Print: shape of X_train, X_test, and the Outcome class ratio in each set.
#      (Check: y_train.mean() ≈ y_test.mean(), stratification ensures this.)
#   4. Fit StandardScaler on X_train ONLY:
#        scaler = StandardScaler()
#        X_train_sc = pd.DataFrame(scaler.fit_transform(X_train), columns=FEATURE_COLS)
#        X_test_sc  = pd.DataFrame(scaler.transform(X_test),      columns=FEATURE_COLS)
#      Note: .fit_transform() on train, .transform() on test NEVER fit on test!
#   5. Print the mean of X_train_sc  should be ~0 for all columns.

# YOUR CODE HERE


In [ ]:
# Provided: K-Fold Cross-Validation, a more robust alternative to a single train/test split
#
# From the lecture (Week 2 Day 2, 'Methods for Model Evaluation'):
#   Train-Test Split → one split, fast, but the result depends on which rows
#                       happened to land in the test set (luck of the draw).
#   K-Fold CV        → the data is split into k folds; the model is trained and
#                       tested k times, each fold gets a turn as the test set.
#                       Results are averaged → a more stable, less luck-dependent estimate.
#                       Common choices: k = 5 or k = 10 (we use k = 5 here).
#
# We evaluate the SAME Linear Regression / REG_FEATURES setup used in Task 11,
# so the comparison between a single split and K-Fold CV is apples-to-apples.

from sklearn.model_selection import cross_val_score, KFold
from sklearn.linear_model import LinearRegression

REG_FEATURES_CV = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
                    'Insulin', 'DiabetesPedigreeFunction', 'Age']
X_cv = df[REG_FEATURES_CV]
y_cv = df['BMI']

kf = KFold(n_splits=5, shuffle=True, random_state=42)  # same RANDOM_STATE=42 as everywhere else

cv_r2_scores = cross_val_score(LinearRegression(), X_cv, y_cv, cv=kf, scoring='r2')

print('--- 5-Fold Cross-Validation: Linear Regression predicting BMI ---')
print(f'R² per fold : {np.round(cv_r2_scores, 3)}')
print(f'Mean R²     : {cv_r2_scores.mean():.3f}')
print(f'Std  R²     : {cv_r2_scores.std():.3f}  (spread across folds = how lucky/unlucky a single split could be)')
print()
print('Compare this Mean R² (and its spread) to the single train/test split R² you will compute in Task 11.')
print('A single split gives ONE number; K-Fold CV gives a DISTRIBUTION, which is the more honest picture.')


### ✍️ Interpretation: Task 10

1. Did the Pearson filter method and RFE (wrapper method) agree on the top 4 features? If they disagreed on any feature, what does this tell you about the limitation of purely linear filter methods?
2. Explain in one sentence why you must fit the StandardScaler only on the training set, not on the full dataset.
3. The lecture described embedded methods (Lasso, Ridge, tree importance) as a third category. How does Lasso differ from RFE in the way it performs feature selection?
4. Compare the single train/test split R² (Task 11) to the 5-Fold CV mean R² above. Are they close? What does the CV standard deviation tell you about how much you should trust a single train/test split?

> *Your answer here:*

---
# Task 11: Linear Regression (Fitting & Interpreting Coefficients)
Regression models the relationship between features and a continuous target. Here we predict **BMI** from the clinical features, a regression task, even though the dataset was originally framed as classification.

> **Goal:** Fit Simple and Multiple Linear Regression models predicting BMI, interpret each coefficient, visualise the regression line and residuals.

In [ ]:
# ── Task 11 Setup (do not modify) ─────────────────────────────────────────────

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 4)

URL = 'https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv'
df_raw = pd.read_csv(URL)
SUSPECT_COLS = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df = df_raw.copy()
df[SUSPECT_COLS] = df[SUSPECT_COLS].replace(0, np.nan)
df[SUSPECT_COLS] = df[SUSPECT_COLS].fillna(df[SUSPECT_COLS].median())

# Regression target: BMI (continuous, this is a regression task).
# Predictors: all other clinical features except Outcome and BMI itself.
REG_FEATURES = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
                'Insulin', 'DiabetesPedigreeFunction', 'Age']
REG_TARGET   = 'BMI'

X_reg = df[REG_FEATURES]
y_reg = df[REG_TARGET]

# Same 80/20 split as Task 10 (same random_state=42 for reproducibility across modules).
X_tr, X_te, y_tr, y_te = train_test_split(
    X_reg, y_reg, test_size=0.20, random_state=42
)

print(f'Train: {len(X_tr)} rows  |  Test: {len(X_te)} rows')
print('Ready')


In [ ]:
# Provided: Simple Linear Regression, predict BMI from Glucose only

# --- Why Simple LR first? ---
# Starting with one predictor makes the model completely visualisable (scatter + line).
# It also establishes a baseline: if Multiple LR barely improves over Simple LR,
# the extra features are not contributing much.

# sklearn requires X to be 2-D, double brackets [[...]] give shape (n, 1) instead of (n,).
X_tr_slr = X_tr[['Glucose']]   # train set, one column
X_te_slr = X_te[['Glucose']]   # test set, one column

slr = LinearRegression()          # create the model object
slr.fit(X_tr_slr, y_tr)          # fit ONLY on the training set

# intercept_ = β₀: predicted BMI when Glucose = 0 (algebraically necessary; not clinically meaningful here)
# coef_[0]   = β₁: expected change in BMI per 1 mg/dL increase in Glucose
print(f'Intercept  β₀ = {slr.intercept_:.4f}')
print(f'Slope      β₁ = {slr.coef_[0]:.4f}')
print(f'Equation   : BMI = {slr.intercept_:.2f} + {slr.coef_[0]:.4f} × Glucose')
print()

# Evaluate on the TEST set, this is the honest performance estimate.
y_pred_slr = slr.predict(X_te_slr)
mae  = mean_absolute_error(y_te, y_pred_slr)
rmse = np.sqrt(mean_squared_error(y_te, y_pred_slr))
r2   = r2_score(y_te, y_pred_slr)
print(f'--- Simple LR Test-Set Performance ---')
print(f'MAE  = {mae:.4f}  (average absolute error in BMI units)')
print(f'RMSE = {rmse:.4f}  (root mean squared error, penalises large errors more)')
print(f'R²   = {r2:.4f}  (fraction of BMI variance explained by Glucose alone)')

# --- Visualise: scatter + regression line + residuals ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: scatter plot with the fitted line.
axes[0].scatter(X_te_slr, y_te, alpha=0.4, s=20, color='steelblue', label='Test data')
x_line = np.linspace(X_te_slr.min().values[0], X_te_slr.max().values[0], 200).reshape(-1,1)
axes[0].plot(x_line, slr.predict(x_line), color='red', linewidth=2, label='Regression line')
axes[0].set_xlabel('Glucose (mg/dL)')
axes[0].set_ylabel('BMI')
axes[0].set_title(f'Simple LR: Glucose → BMI  (Test R²={r2:.3f})')
axes[0].legend()

# Right: residual plot, residuals vs predicted values.
# A good model has residuals randomly scattered around 0 with no visible pattern.
# Patterns (curved, funnel-shaped) suggest model misspecification or unequal variance.
residuals = y_te.values - y_pred_slr
axes[1].scatter(y_pred_slr, residuals, alpha=0.4, s=20, color='coral')
axes[1].axhline(0, color='black', linewidth=1.2, linestyle='--')
axes[1].set_xlabel('Predicted BMI')
axes[1].set_ylabel('Residual (Actual, Predicted)')
axes[1].set_title('Residual Plot, should look like random noise around 0')

plt.tight_layout()
plt.show()


In [ ]:
# TODO: Multiple Linear Regression, predict BMI from ALL REG_FEATURES.
#
# Steps:
#   1. Fit a LinearRegression() model on X_tr and y_tr (all 7 features).
#   2. Build a coefficient table:
#      Create a DataFrame with columns 'Feature' and 'Coefficient',
#      using REG_FEATURES and mlr.coef_. Sort by absolute coefficient descending.
#      Print the table along with the intercept.
#
#   3. Evaluate on the TEST set (X_te, y_te):
#      Print MAE, RMSE, and R². Compare to the Simple LR results above.
#
#   4. Create the 'Predicted vs Actual' plot:
#      - x-axis: y_te (actual BMI)
#      - y-axis: predictions from Multiple LR
#      - Add a red dashed diagonal line: 'Perfect prediction' (y = x)
#      - Title: f'Multiple LR: Actual vs Predicted BMI  (R²={r2:.3f})'
#
#   5. Create the coefficient bar chart:
#      Horizontal bar chart of all 7 coefficients, sorted by value.
#      Positive bars → feature increases BMI prediction; negative bars → decreases it.

# YOUR CODE HERE


### ✍️ Interpretation: Task 11

1. The Simple LR slope β₁ for Glucose→BMI is probably small and positive. Write one sentence interpreting this coefficient in plain English (e.g. 'For every 1 mg/dL increase in Glucose...').
2. Did Multiple LR improve R² compared to Simple LR? If the improvement was small, what does this tell you about the relationship between the extra features and BMI?
3. Looking at the residual plot from Simple LR: is the assumption of *constant variance* (homoscedasticity) satisfied? Describe what you see in the plot.

> *Your answer here:*

---
# Task 12: Regression Evaluation Metrics & Residual Analysis
A single R² number hides a lot of information. Residual analysis studying the pattern of errors reveals whether the model's assumptions are satisfied and where it fails.

> **Goal:** Compute all four regression metrics (MAE, MSE, RMSE, R²), compare a linear and a polynomial model, and perform a structured residual analysis.

In [ ]:
# ── Task 12 Setup (do not modify) ─────────────────────────────────────────────

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
# Pipeline chains preprocessing steps and a model into one object.
# This prevents data leakage: the polynomial transformer is fitted only on
# the training fold, not on the full dataset.
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 4)

URL = 'https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv'
df_raw = pd.read_csv(URL)
SUSPECT_COLS = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df = df_raw.copy()
df[SUSPECT_COLS] = df[SUSPECT_COLS].replace(0, np.nan)
df[SUSPECT_COLS] = df[SUSPECT_COLS].fillna(df[SUSPECT_COLS].median())

# Use only Glucose (one feature) so we can visualise the fit in 2-D.
# With two features we would need a 3-D plot, much harder to interpret.
X_1d = df[['Glucose']]
y_1d = df['BMI']

X_tr_1d, X_te_1d, y_tr_1d, y_te_1d = train_test_split(
    X_1d, y_1d, test_size=0.20, random_state=42
)

print(f'Train: {len(X_tr_1d)} rows  |  Test: {len(X_te_1d)} rows')
print('Ready')


In [ ]:
# Provided: compute all four evaluation metrics and explain each one

# Fit a simple linear model (degree=1) our baseline for the comparison.
lin_model = LinearRegression()
lin_model.fit(X_tr_1d, y_tr_1d)
y_pred_lin = lin_model.predict(X_te_1d)

# --- Metric 1: MAE (Mean Absolute Error) ---
# Average of |actual - predicted| across all test rows.
# Interpretation: on average, our prediction is off by MAE BMI units.
# Advantage: same units as the target; easy to explain to non-technical audiences.
mae  = mean_absolute_error(y_te_1d, y_pred_lin)

# --- Metric 2: MSE (Mean Squared Error) ---
# Average of (actual - predicted)² across all test rows.
# Penalises large errors MORE than small ones (squaring amplifies them).
# Disadvantage: units are squared (e.g. BMI²) not directly interpretable.
mse  = mean_squared_error(y_te_1d, y_pred_lin)

# --- Metric 3: RMSE (Root Mean Squared Error) ---
# Square root of MSE restores the original units (BMI).
# More sensitive to large errors than MAE; the most commonly reported metric.
rmse = np.sqrt(mse)

# --- Metric 4: R² (Coefficient of Determination) ---
# R² = 1 - (SS_residual / SS_total)
# Interpretation: 'the model explains R²% of the variance in BMI.'
# R² = 1: perfect predictions. R² = 0: model is no better than predicting the mean.
# R² can be negative if the model is worse than predicting the mean.
r2   = r2_score(y_te_1d, y_pred_lin)

print('--- Linear Regression Evaluation Metrics (Test Set) ---')
print(f'MAE  = {mae:.4f}  → average absolute prediction error in BMI units')
print(f'MSE  = {mse:.4f}  → average squared error (BMI²) penalises large errors heavily')
print(f'RMSE = {rmse:.4f}  → back in BMI units; typical prediction error ≈ {rmse:.1f} BMI points')
print(f'R²   = {r2:.4f}  → the model explains {r2*100:.1f}% of BMI variance')


In [ ]:
# TODO: Fit a Polynomial Regression (degree=2) and compare it to Linear Regression.
#
# Polynomial regression captures curved relationships.
# Instead of: BMI = β₀ + β₁·Glucose
# It fits: BMI = β₀ + β₁·Glucose + β₂·Glucose²
#
# We use a sklearn Pipeline to safely chain feature expansion and fitting.
#
# Steps:
#   1. Build the pipeline:
#        poly_pipe = Pipeline([
#            ('poly',  PolynomialFeatures(degree=2, include_bias=False)),
#            ('model', LinearRegression())
#        ])
#      include_bias=False: the LinearRegression step already adds its own intercept.
#
#   2. Fit poly_pipe on X_tr_1d / y_tr_1d.
#   3. Predict on X_te_1d → y_pred_poly.
#   4. Compute and print MAE, RMSE, R² for the polynomial model.
#   5. Build a comparison table:
#      'Model' | 'MAE' | 'RMSE' | 'R²'
#      with rows for 'Linear (degree=1)' and 'Polynomial (degree=2)'. Round to 4 dp.
#
#   6. Plot BOTH regression curves on the same scatter plot:
#        - Create a dense x_line: np.linspace(X_1d.min(), X_1d.max(), 300).reshape(-1,1)
#        - Plot lin_model.predict(x_line) in red: 'Linear'
#        - Plot poly_pipe.predict(x_line) in green: 'Polynomial degree=2'
#        - Scatter the test data points in blue.
#        - Add title, axis labels, and legend.

# YOUR CODE HERE


In [ ]:
# TODO: Residual analysis: interpret the error patterns of both models.
#
# A residual = actual - predicted.  For a well-fitted model:
#   - Residuals should be randomly scattered around 0 (no pattern).
#   - Residuals should have roughly constant variance across all predicted values.
#   - Residuals should be approximately Normally distributed.
#
# From the lecture table (slide 66):
#   Pattern                  → Diagnosis
#   Random around 0          → Model is reasonable
#   Curved pattern           → Linear model too simple
#   Increasing spread        → Unequal error variance (heteroscedasticity)
#   Large isolated points    → Possible outliers
#   Clear systematic pattern → Missing features or wrong model form
#
# Steps:
#   1. Compute residuals for both models:
#        resid_lin  = y_te_1d.values - y_pred_lin
#        resid_poly = y_te_1d.values - y_pred_poly
#
#   2. Create a 2×2 grid of plots (plt.subplots(2, 2, figsize=(13, 9))):
#      Row 1 = Linear model, Row 2 = Polynomial model.
#      Col 1: Residuals vs Predicted values (scatter + axhline at 0).
#      Col 2: Histogram of residuals (bins=30, edgecolor='white').
#
#   3. For each histogram, also compute stats.skew(residuals) and print it.
#      A skewness close to 0 indicates approximately Normal residuals.
#
#   4. Add a main title: 'Residual Analysis: Linear vs Polynomial Regression'.

# YOUR CODE HERE


### ✍️ Interpretation: Task 12

1. Did the Polynomial (degree=2) model outperform the Linear model on R²? If the improvement was small, what does this suggest about the relationship between Glucose and BMI?
2. Looking at the residual plots: do the residuals appear randomly scattered, or is there a visible pattern? Based on the lecture's residual pattern table, what does the pattern (if any) indicate?
3. The lecture noted that a curved residual pattern means 'the linear model may be too simple.' If you saw a strong curved pattern, what would be your next modelling step?

> *Your answer here:*

---
## 📋 Module D: Handoff Summary

Complete this table before merging with your group.

| Item | Your finding |
|---|---|
| Top 4 features by Pearson |r| | ???, ???, ???, ??? |
| Top 4 features by RFE | ???, ???, ???, ??? |
| Did filter and wrapper methods agree? | Yes / No ??? |
| 5-Fold CV mean R² (BMI regression) | ??? (± ???) |
| Simple LR (Glucose→BMI): β₁, R² | β₁=???, R²=??? |
| Multiple LR (all features→BMI): R² | ??? |
| Linear model: MAE, RMSE | MAE=???, RMSE=??? |
| Polynomial (deg=2) model: R² | ??? |
| Residual pattern observed | ??? |

> Fill in the table above before handing off to the group.